# pathlib — praktyczny przewodnik: od podstaw po zastosowania w pracy analityka

`pathlib` to obiektowy sposób pracy ze ścieżkami — zastępuje `os.path`
(operacje na stringach) obiektami `Path`, które: działają identycznie na
Windows/Linux/macOS (różne separatory ścieżek obsłużone pod spodem),
wspierają operator `/` do składania ścieżek, i są **akceptowane
bezpośrednio** przez `pandas`, `polars`, `duckdb`, `open()` — nigdzie nie
musisz ręcznie konwertować do `str`.

Część standardowej biblioteki — nic do instalacji.

In [1]:
from pathlib import Path
from datetime import datetime, timedelta
import os
import pandas as pd

## 1. Podstawy — bieżący folder i tworzenie ścieżek

- `Path.cwd()` — bieżący katalog roboczy (ten, z którego uruchomiony jest
  proces — w notebooku zwykle katalog, w którym leży plik `.ipynb`).
- `Path.home()` — katalog domowy użytkownika.
- `Path("cokolwiek")` — ścieżka względna albo bezwzględna, w zależności od
  podanego stringa; sama w sobie **nie sprawdza**, czy coś tam istnieje.
- `str(sciezka)` — konwersja z powrotem do stringa, gdy naprawdę tego
  potrzebujesz (np. f-string do komunikatu).

In [2]:
print("cwd:", Path.cwd())
print("home:", Path.home())

wzgledna = Path("dane/eksport.csv")
bezwzgledna = Path.cwd() / "dane" / "eksport.csv"

print("względna:", wzgledna, "| bezwzględna:", wzgledna.is_absolute())
print("bezwzględna:", bezwzgledna, "| bezwzględna:", bezwzgledna.is_absolute())

cwd: /home/claude
home: /root
względna: dane/eksport.csv | bezwzględna: False
bezwzględna: /home/claude/dane/eksport.csv | bezwzględna: True


## 2. Składanie ścieżek

Operator `/` to podstawowy, najczytelniejszy sposób — działa też z wieloma
segmentami naraz. `.joinpath(...)` robi to samo, przydaje się głównie, gdy
segmenty budujesz programowo (np. rozpakowany `*parts`).

In [3]:
projekt = Path.cwd() / "projekt_demo"

sciezka1 = projekt / "dane" / "raw" / "plik.csv"          # operator / (zalecane)
sciezka2 = projekt.joinpath("dane", "raw", "plik.csv")     # równoważne

print(sciezka1)
print(sciezka1 == sciezka2)

/home/claude/projekt_demo/dane/raw/plik.csv
True


## 3. Rozbijanie ścieżki na części

| Atrybut | Zwraca |
|---|---|
| `.name` | pełna nazwa pliku z rozszerzeniem (`eksport_2025.csv`) |
| `.stem` | nazwa bez ostatniego rozszerzenia (`eksport_2025`) |
| `.suffix` | ostatnie rozszerzenie z kropką (`.csv`) |
| `.suffixes` | lista wszystkich rozszerzeń — przydatne dla `.tar.gz` itp. |
| `.parent` | katalog nadrzędny (jeden poziom w górę), jako `Path` |
| `.parents[i]` | **sekwencja** przodków; `parents[0]` = `.parent`, `parents[1]` = dziadek, itd. |
| `.parts` | wszystkie segmenty ścieżki jako krotka stringów |

In [4]:
p = Path.cwd() / "projekt_demo" / "dane" / "raw" / "eksport_2025.tar.gz"

print("name    :", p.name)
print("stem    :", p.stem)          # tylko ostatnie rozszerzenie ucięte -> 'eksport_2025.tar'
print("suffix  :", p.suffix)
print("suffixes:", p.suffixes)
print("parent  :", p.parent)
print("parents[0]:", p.parents[0])   # == p.parent
print("parents[1]:", p.parents[1])   # katalog wyżej
print("parents[2]:", p.parents[2])
print("parts   :", p.parts)

name    : eksport_2025.tar.gz
stem    : eksport_2025.tar
suffix  : .gz
suffixes: ['.tar', '.gz']
parent  : /home/claude/projekt_demo/dane/raw
parents[0]: /home/claude/projekt_demo/dane/raw
parents[1]: /home/claude/projekt_demo/dane
parents[2]: /home/claude/projekt_demo
parts   : ('/', 'home', 'claude', 'projekt_demo', 'dane', 'raw', 'eksport_2025.tar.gz')


> `.stem` ucina tylko **ostatnie** rozszerzenie — dla `plik.tar.gz` dostaniesz
> `plik.tar`, nie `plik`. Przy takich podwójnych rozszerzeniach czasem
> wygodniej wziąć `p.name.split(".")[0]` albo zdjąć oba suffixy po kolei
> (`Path(p.stem).stem`).

## 4. Sprawdzanie, co kryje się pod ścieżką

`exists()`, `is_file()`, `is_dir()` **faktycznie odpytują system plików** —
w odróżnieniu od atrybutów z sekcji 3, które są czystą manipulacją
stringiem i działają nawet dla ścieżki, która nigdzie nie istnieje.

In [5]:
projekt = Path.cwd() / "projekt_demo"
projekt.mkdir(exist_ok=True)   # żeby poniższe sprawdzenia miały na czym pracować

print("projekt istnieje?      ", projekt.exists())
print("projekt to folder?     ", projekt.is_dir())
print("projekt to plik?       ", projekt.is_file())
print("nieistniejąca ścieżka: ", (projekt / "nie_ma_takiego.csv").exists())
print("ścieżka bezwzględna?   ", projekt.is_absolute())

projekt istnieje?       True
projekt to folder?      True
projekt to plik?        False
nieistniejąca ścieżka:  False
ścieżka bezwzględna?    True


## 5. Tworzenie folderów i podfolderów

`mkdir(parents=True, exist_ok=True)` — praktyczny domyślny wariant:
- `parents=True` — tworzy też brakujące katalogi nadrzędne (bez tego
  `mkdir()` rzuci błąd, jeśli rodzic nie istnieje),
- `exist_ok=True` — nie wywala się, jeśli katalog już jest (bez tego
  powtórne uruchomienie skryptu/notebooka rzuci `FileExistsError`).

In [6]:
for podfolder in ["dane/raw", "dane/processed", "output", "logi"]:
    (projekt / podfolder).mkdir(parents=True, exist_ok=True)

sorted(str(p.relative_to(projekt)) for p in projekt.rglob("*") if p.is_dir())

['dane', 'dane/processed', 'dane/raw', 'logi', 'output']

## 6. Filtrowanie plików w folderach

- **`iterdir()`** — wszystkie elementy jednego poziomu (pliki i foldery),
  bez wchodzenia w podfoldery.
- **`glob(wzorzec)`** — dopasowanie wg wzorca (`*`, `?`, `[...]`) na jednym
  poziomie; `**` w ścieżce włącza rekurencję.
- **`rglob(wzorzec)`** — skrót na `glob("**/" + wzorzec)`, czyli
  rekurencyjnie przez wszystkie podfoldery.

Najpierw przygotujmy przykładowe pliki do przefiltrowania w dalszych
sekcjach.

In [7]:
raw = projekt / "dane" / "raw"

# kilka plików "na teraz" + kilka celowo cofniętych w czasie (symulacja
# historii eksportów z różnych dni — do demonstracji sekcji 7-9)
teraz = datetime.now()
wiek_pliku_w_dniach = {
    "eksport_dzis.csv": 0.1,
    "eksport_wczoraj.csv": 1,
    "eksport_3dni.csv": 3,
    "eksport_6dni.csv": 6,
    "eksport_10dni.csv": 10,
    "eksport_30dni.csv": 30,
}

for nazwa, dni_wstecz in wiek_pliku_w_dniach.items():
    plik = raw / nazwa
    plik.write_text(f"id,wartosc\n1,{dni_wstecz}\n", encoding="utf-8")
    czas = (teraz - timedelta(days=dni_wstecz)).timestamp()
    os.utime(plik, (czas, czas))   # ustawia czas dostępu i modyfikacji na przeszłość

(raw / "notatki.txt").write_text("nie-csv, do odfiltrowania", encoding="utf-8")

sorted(p.name for p in raw.iterdir())

['eksport_10dni.csv',
 'eksport_30dni.csv',
 'eksport_3dni.csv',
 'eksport_6dni.csv',
 'eksport_dzis.csv',
 'eksport_wczoraj.csv',
 'notatki.txt']

In [8]:
# iterdir() — wszystko na jednym poziomie, trzeba samemu filtrować
wszystkie_pliki = [p for p in raw.iterdir() if p.is_file()]

# glob() — filtr wzorcem od razu we wywołaniu
csv_glob = list(raw.glob("*.csv"))

# rglob() — rekurencyjnie przez wszystkie podfoldery (tu bez różnicy,
# bo pliki są płasko w jednym folderze, ale przy zagnieżdżonej strukturze
# to jedyny sposób, żeby złapać pliki z podfolderów bez ręcznej pętli)
csv_rglob = list((projekt / "dane").rglob("*.csv"))

print("iterdir (wszystko):", len(wszystkie_pliki))
print("glob *.csv         :", len(csv_glob))
print("rglob *.csv (rekur.):", len(csv_rglob))

iterdir (wszystko): 7
glob *.csv         : 6
rglob *.csv (rekur.): 6


In [9]:
# filtrowanie po własnym warunku — zwykła list comprehension po iterdir()/glob()
duze_nazwy = [p for p in raw.glob("*.csv") if "eksport" in p.stem and int(p.stat().st_size) > 0]
bez_notatek = [p for p in raw.iterdir() if p.suffix == ".csv"]   # odpowiednik glob("*.csv") pisany ręcznie

print(len(duze_nazwy), len(bez_notatek))

6 6


## 7. Pliki zmodyfikowane w ostatnich 24h / 7 dniach

Data modyfikacji siedzi w `.stat().st_mtime` — timestamp Unix (float,
sekundy). Porównujesz go z progiem czasowym liczonym jako `datetime.now() -
timedelta(...)`, skonwertowanym też do timestampu (`.timestamp()`) — szybciej
niż konwertować `st_mtime` na `datetime` dla każdego pliku, gdy filtrujesz
dużo plików.

In [10]:
próg_24h = (datetime.now() - timedelta(hours=24)).timestamp()
próg_7d = (datetime.now() - timedelta(days=7)).timestamp()

pliki_24h = [p for p in raw.glob("*.csv") if p.stat().st_mtime >= próg_24h]
pliki_7d = [p for p in raw.glob("*.csv") if p.stat().st_mtime >= próg_7d]

print("ostatnie 24h:", sorted(p.name for p in pliki_24h))
print("ostatnie 7 dni:", sorted(p.name for p in pliki_7d))

ostatnie 24h: ['eksport_dzis.csv']
ostatnie 7 dni: ['eksport_3dni.csv', 'eksport_6dni.csv', 'eksport_dzis.csv', 'eksport_wczoraj.csv']


> `st_mtime` to czas **modyfikacji zawartości**. `st_ctime` na Linuksie to
> czas zmiany metadanych (nie utworzenia!), a na Windows — czas utworzenia.
> `st_atime` to czas ostatniego odczytu. Do "kiedy plik został wygenerowany/
> nadpisany" prawie zawsze chcesz `st_mtime` — jest spójny między systemami.

## 8. Ostatni zmodyfikowany plik

`max(..., key=...)` po `st_mtime` — nie trzeba sortować całej listy, żeby
znaleźć jeden element.

In [11]:
ostatni_plik = max(raw.glob("*.csv"), key=lambda p: p.stat().st_mtime)

print("najnowszy plik:", ostatni_plik.name)
print("zmodyfikowany :", datetime.fromtimestamp(ostatni_plik.stat().st_mtime))

najnowszy plik: eksport_dzis.csv
zmodyfikowany : 2026-09-23 08:03:17.514269


## 9. Ostatnie N plików — pobranie i połączenie w jeden DataFrame

Wzorzec: posortuj malejąco po `st_mtime`, weź pierwsze `n`, wczytaj i
skonkatenuj. Przydaje się np. do "weź 5 ostatnich dziennych eksportów i
policz z nich trend".

In [12]:
n = 3
ostatnie_n = sorted(raw.glob("*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)[:n]

print("wybrane pliki (od najnowszego):", [p.name for p in ostatnie_n])

# każdy plik wczytany osobno + kolumna źródłowa (przydatna do debugowania,
# skąd wzięła się dana partia danych po konkatenacji)
ramki = []
for plik in ostatnie_n:
    df = pd.read_csv(plik)
    df["zrodlo_plik"] = plik.name
    ramki.append(df)

polaczone = pd.concat(ramki, ignore_index=True)
polaczone

wybrane pliki (od najnowszego): ['eksport_dzis.csv', 'eksport_wczoraj.csv', 'eksport_3dni.csv']


,id,wartosc,zrodlo_plik
0,1,0.1,eksport_dzis.csv
1,1,1.0,eksport_wczoraj.csv
2,1,3.0,eksport_3dni.csv


## 10. Zapisywanie plików

- **Tekst/CSV ręcznie**: `.write_text(tresc, encoding="utf-8")` — zawsze
  podawaj `encoding` jawnie, żeby nie zależeć od domyślnego kodowania
  systemu (na Windows bywa inne niż UTF-8).
- **Binaria**: `.write_bytes(dane)`.
- **DataFrame**: `pandas`/`polars` przyjmują `Path` bezpośrednio jako
  argument `to_csv()`/`write_parquet()` — nie trzeba `str(sciezka)`.

In [13]:
output = projekt / "output"

(output / "log.txt").write_text("start przetwarzania\n", encoding="utf-8")

polaczone.to_csv(output / "polaczone.csv", index=False)   # Path wprost, bez str()

list(output.iterdir())

[PosixPath('/home/claude/projekt_demo/output/log.txt'),
 PosixPath('/home/claude/projekt_demo/output/polaczone.csv')]

## 11. Zapisywanie plików z datą w nazwie

Format daty wprost w f-stringu jest najprostszy. `with_name()` i
`with_suffix()` przydają się, gdy bazujesz na istniejącej ścieżce i
podmieniasz tylko fragment (np. dopisujesz datę do nazwy pliku wejściowego,
zachowując resztę ścieżki i rozszerzenie).

In [14]:
znacznik_czasu = datetime.now().strftime("%Y%m%d_%H%M")

# wariant 1: budowa nazwy od zera
plik_z_data = output / f"eksport_{znacznik_czasu}.csv"
polaczone.to_csv(plik_z_data, index=False)

# wariant 2: dopisanie znacznika do istniejącej nazwy, z zachowaniem folderu i rozszerzenia
plik_zrodlowy = raw / "eksport_dzis.csv"
plik_z_data_v2 = plik_zrodlowy.with_name(f"{plik_zrodlowy.stem}_{znacznik_czasu}{plik_zrodlowy.suffix}")

print(plik_z_data.name)
print(plik_z_data_v2)

eksport_20260923_1027.csv
/home/claude/projekt_demo/dane/raw/eksport_dzis_20260923_1027.csv


## 12. Inne przydatne rzeczy dla analityka

- **`resolve()`** — zwraca pełną, bezwzględną ścieżkę i rozwiązuje `..`/`.`
  — przydatne w logach/komunikatach błędów, żeby nie zgadywać, "względem
  czego" jest ścieżka.
- **`relative_to(inna_sciezka)`** — odwrotność: ścieżka względem podanego
  punktu (użyte już w sekcji 5, żeby nie drukować pełnych ścieżek).
- **`with_suffix(".parquet")`** — podmiana rozszerzenia z zachowaniem
  reszty ścieżki; częsty wzorzec: "zapisz wynik obok źródła, ale jako
  Parquet zamiast CSV".
- **`rename()` / `unlink()` / `rmdir()`** — przenoszenie/usuwanie; dla
  niepustego folderu `rmdir()` rzuci błąd — do rekurencyjnego usuwania
  całego drzewa katalogów służy `shutil.rmtree()`, nie `pathlib`.
- **Sumaryczny rozmiar/najcięższe pliki w folderze** — częste zadanie przy
  sprzątaniu katalogu z eksportami/logami.

In [15]:
# resolve() vs relative_to()
wzgledna = Path("projekt_demo/dane/raw")
print("resolve()   :", wzgledna.resolve())
print("relative_to :", (raw / "eksport_dzis.csv").relative_to(projekt))

resolve()   : /home/claude/projekt_demo/dane/raw
relative_to : dane/raw/eksport_dzis.csv


In [16]:
# with_suffix — ten sam plik, inne rozszerzenie, ta sama lokalizacja
zrodlo = output / "polaczone.csv"
cel_parquet = zrodlo.with_suffix(".parquet")
print(cel_parquet)

polaczone.to_parquet(cel_parquet, index=False)
print("zapisany:", cel_parquet.exists())

/home/claude/projekt_demo/output/polaczone.parquet


zapisany: True


In [17]:
# łączny rozmiar folderu + 3 najcięższe pliki — typowy check przed sprzątaniem/backupem
pliki = [p for p in projekt.rglob("*") if p.is_file()]

rozmiar_lacznie_mb = sum(p.stat().st_size for p in pliki) / (1024 ** 2)
najciezsze = sorted(pliki, key=lambda p: p.stat().st_size, reverse=True)[:3]

print(f"łączny rozmiar: {rozmiar_lacznie_mb:.3f} MB, liczba plików: {len(pliki)}")
for p in najciezsze:
    print(f"  {p.relative_to(projekt)}: {p.stat().st_size} B")

łączny rozmiar: 0.003 MB, liczba plików: 11
  output/polaczone.parquet: 2379 B
  output/eksport_20260923_1027.csv: 95 B
  output/polaczone.csv: 95 B


> **Dlaczego `Path`, a nie `os.path`?** `os.path` operuje na gołych
> stringach (`os.path.join(a, b)`, `os.path.splitext(p)`) — działa, ale
> szybko robi się nieczytelne przy łańcuchu operacji. `Path` daje to samo
> jako metody/atrybuty na jednym obiekcie (`p.parent.parent / "inny.csv"`),
> **jest przenośny między systemami operacyjnymi bez żadnego wysiłku z
> Twojej strony**, i — praktycznie najważniejsze na co dzień — cały
> ekosystem (`pandas`, `polars`, `duckdb`, `open()`) przyjmuje go
> bezpośrednio, bez `str(...)` na każdym wywołaniu.

In [18]:
# porządki po demie
import shutil
shutil.rmtree(projekt, ignore_errors=True)
print("posprzątane:", not projekt.exists())

posprzątane: True
